In [4]:
from fractal_printer.mesh import fractal_sdfs as fs, mesh_generation as mg
import numpy as np
from importlib import reload
import quaternion
from sdf import sdf
import matplotlib.pyplot as plt
from fractal_printer.paths import OUTPUT_DIR

In [5]:
reload(mg)
reload(fs)

settings = {"power": 7,
 "cx": 0.07799999999999985,
 "cy": -0.784,
 "cz": -0.745,
 "cw": 0.0,
 "slice": 0.0,
 "offset": 0.002,
 "iterations": 44,
 "bailout": 30}

# Generate the points
mesh = mg.generate_mesh(
    fs.quaternion_julia_sdf(**settings),
    samples = 2**28,
    simplify = None,
    bounds = mg.box_bounds(size=3.5),
    save_path = OUTPUT_DIR / "high_order_test.ply"
)


min -1.75, -1.75, -1.75
max 1.75, 1.75, 1.75
step 0.00542569, 0.00542569, 0.00542569
295408296 samples in 9261 batches with 8 workers


KeyboardInterrupt: 

In [6]:
# Generate a random order-5 juilia
reload(mg)
reload(fs)
from functools import partial


gen = np.random.default_rng(2)
order = 3
C = quaternion.as_quat_array(gen.standard_normal((order-1,4))*0.5)
C = C / np.abs(C)

print(C)

high_order_julia = fs.general_julia_sdf(
    update = partial(fs.polynomial_update, C = C),
    order = order,
    offset = 0.0025,
)

slab = sdf.slab(z0 = 0)

# Generate the points
mesh = mg.generate_mesh(
    high_order_julia & slab,
    samples = 2**22,
    simplify = 0.1,
    bounds = mg.box_bounds(size=3.5),
    save_path = OUTPUT_DIR / "high_order_test_cutoff.ply"
)

[quaternion(0.074495203948912, -0.205985480887157, -0.162764890090382, -0.962043677182155)
 quaternion(0.785251894618089, 0.499224722530309, -0.141989137598643, 0.337628824624366)]
min -1.75, -1.75, -1.75
max 1.75, 1.75, 1.75
step 0.0217027, 0.0217027, 0.0217027
4657463 samples in 216 batches with 8 workers
  100% (216 of 216) [##############################] 0:00:10 0:00:00    
120 skipped, 72 empty, 24 nonempty
110772 triangles in 10.4664 seconds
Simplifying mesh by 0.1x ...
Saving mesh to C:\Users\owenb\Code\personal\fractal-printer\outputs\high_order_test_cutoff.ply...


In [ ]:
def polynomial_update(z, C):
    order = len(C)+1
    print(order)
    z_1 = np.power(z, order)
    for i, c in enumerate(C):
        print("\t",i)
        z_1 = z_1 + np.power(z, order-1-i) * c
    return z_1

order = 3
C = quaternion.as_quat_array(np.random.randn(order-1,4))*0.5

print(C.shape)
update = partial(polynomial_update, C=C)

foo = quaternion.as_quat_array(np.random.randn(10,4))
update(foo)

(2,)
3
	 0
	 1


array([quaternion(-5.40607331986989, 2.81123490872448, -0.970676604717005, -3.54986869124246),
       quaternion(-2.41334709709985, -0.0495524729465072, -1.28438784624023, -0.910711137878433),
       quaternion(0.0098991148428127, -1.21085743135569, -0.798302434318201, -1.1441709796793),
       quaternion(-2.85010738256141, -3.51423774448331, -18.1036838497106, -2.78701970282583),
       quaternion(0.501273795426748, 0.995415810331766, -1.49595269150803, 0.626873654732289),
       quaternion(-1.18276982700419, 1.86165584545554, 2.30543736852067, 2.38339126417981),
       quaternion(1.87320496530083, 0.0167839115207986, 0.439488626964463, -1.26839266635583),
       quaternion(2.51320630174386, -0.504083208301348, -0.423168532496961, -3.74989458917984),
       quaternion(-20.942815836463, -8.53768587800247, 4.64557616919294, -7.75484410381538),
       quaternion(-0.103721243269284, -0.152938288258699, -0.250024640324056, 0.0470152467196325)],
      dtype=quaternion)